In [1]:
import pandas as pd
import numpy as np
import pdfplumber
import re
from pathlib import Path

print("Inventory notebook ready")

Inventory notebook ready


In [2]:
inventory_pdf = Path("../data/raw/inventory_count.pdf")

print("Inventory PDF exists:", inventory_pdf.exists())

Inventory PDF exists: True


In [3]:
inventory_text = ""

with pdfplumber.open(inventory_pdf) as pdf:
    print("Number of pages:", len(pdf.pages))
    
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            inventory_text += text + "\n"

print(inventory_text[:1000])

Number of pages: 49
(cid:13)
06/24/26 Inventory Count Sheet Page 1
Item No Description Quantity Count Item No Description Quantity Count
------------ ------------------------------ -------- ------------ ------------ ------------------------------ -------- ------------
33412 10 BARRELL PUB FRUIT PUNCH 12N 4.75____________ 228CR A-TREAT 12 OZ CREAM BOTTLES CS .00____________
33413 10 BARRELL PUB ORIGINAL 12NR C 4.75____________ 22820BB A-TREAT BIG BLUE 20OZ BOTTLE 2 .87____________
12476 14TH STAR CITRANADE 16CAN CASE .50____________ 2820BIB A-TREAT BIRCH BEER 20OZ BTTL C .00____________
125101 1809 BERLINER WEISSE 16.9OZ CA .75____________ 22820BC A-TREAT BLACK CHERRY 20OZ 24PK .12____________
10602 1911 BLACK CHERRY 16CAN CASE .66____________ 22820C A-TREAT COLA 20OZ BTTL 24PK CA 1.41____________
70212 1911 BLACK CHERRY 19.2CAN CASE 1.66____________ 2820CR A-TREAT CREAM 20OZ BOTTLE CASE .45____________
70210 1911 CANDY CORN 16CAN CASE .00____________ 2820GR A-TREAT GRAPE 20OZ BOTTLE CA

In [4]:
inventory_lines = inventory_text.split("\n")

print("Total lines:", len(inventory_lines))

for line in inventory_lines[:20]:
    print(line)

Total lines: 2593
(cid:13)
06/24/26 Inventory Count Sheet Page 1
Item No Description Quantity Count Item No Description Quantity Count
------------ ------------------------------ -------- ------------ ------------ ------------------------------ -------- ------------
33412 10 BARRELL PUB FRUIT PUNCH 12N 4.75____________ 228CR A-TREAT 12 OZ CREAM BOTTLES CS .00____________
33413 10 BARRELL PUB ORIGINAL 12NR C 4.75____________ 22820BB A-TREAT BIG BLUE 20OZ BOTTLE 2 .87____________
12476 14TH STAR CITRANADE 16CAN CASE .50____________ 2820BIB A-TREAT BIRCH BEER 20OZ BTTL C .00____________
125101 1809 BERLINER WEISSE 16.9OZ CA .75____________ 22820BC A-TREAT BLACK CHERRY 20OZ 24PK .12____________
10602 1911 BLACK CHERRY 16CAN CASE .66____________ 22820C A-TREAT COLA 20OZ BTTL 24PK CA 1.41____________
70212 1911 BLACK CHERRY 19.2CAN CASE 1.66____________ 2820CR A-TREAT CREAM 20OZ BOTTLE CASE .45____________
70210 1911 CANDY CORN 16CAN CASE .00____________ 2820GR A-TREAT GRAPE 20OZ BOTTLE CASE

In [5]:
inventory_item_lines = []

for line in inventory_lines:
    line = line.strip()
    
    if not line:
        continue
    if "Inventory Count Sheet" in line:
        continue
    if "Item No Description" in line:
        continue
    if "------------" in line:
        continue
    
    inventory_item_lines.append(line)

print("Possible inventory lines:", len(inventory_item_lines))

for line in inventory_item_lines[:10]:
    print(line)

Possible inventory lines: 2448
(cid:13)
33412 10 BARRELL PUB FRUIT PUNCH 12N 4.75____________ 228CR A-TREAT 12 OZ CREAM BOTTLES CS .00____________
33413 10 BARRELL PUB ORIGINAL 12NR C 4.75____________ 22820BB A-TREAT BIG BLUE 20OZ BOTTLE 2 .87____________
12476 14TH STAR CITRANADE 16CAN CASE .50____________ 2820BIB A-TREAT BIRCH BEER 20OZ BTTL C .00____________
125101 1809 BERLINER WEISSE 16.9OZ CA .75____________ 22820BC A-TREAT BLACK CHERRY 20OZ 24PK .12____________
10602 1911 BLACK CHERRY 16CAN CASE .66____________ 22820C A-TREAT COLA 20OZ BTTL 24PK CA 1.41____________
70212 1911 BLACK CHERRY 19.2CAN CASE 1.66____________ 2820CR A-TREAT CREAM 20OZ BOTTLE CASE .45____________
70210 1911 CANDY CORN 16CAN CASE .00____________ 2820GR A-TREAT GRAPE 20OZ BOTTLE CASE 2.91____________
10184 1911 CIDER DONUT 16CAN CASE .00____________ 2820OR A-TREAT ORANGE 20OZ BOTTLE CA .12____________
10695 1911 CRANBERRY CIDER 16 CAN CA .00____________ 2820RB A-TREAT ROOT BEER 200Z BOTTLE .95____________


In [6]:
inventory_records = []

for line in inventory_item_lines:

    match = re.match(r"^(\d+)\s+(.*?)\s+([\d\.]+)_*", line)

    if match:

        item_number = match.group(1)
        description = match.group(2).strip()
        quantity = float(match.group(3))

        inventory_records.append([
            item_number,
            description,
            quantity
        ])

print("Inventory records found:", len(inventory_records))

Inventory records found: 2073


In [7]:
inventory_df = pd.DataFrame(
    inventory_records,
    columns=[
        "item_number",
        "description",
        "quantity_on_hand"
    ]
)

inventory_df.head()

,item_number,description,quantity_on_hand
0,33412,10 BARRELL PUB FRUIT PUNCH,12.0
1,33413,10 BARRELL PUB ORIGINAL,12.0
2,12476,14TH STAR CITRANADE,16.0
3,125101,1809 BERLINER WEISSE,16.9
4,10602,1911 BLACK CHERRY,16.0


In [8]:
inventory_df["item_number"] = inventory_df["item_number"].astype(str)
inventory_df["description"] = inventory_df["description"].str.strip()
inventory_df["quantity_on_hand"] = pd.to_numeric(
    inventory_df["quantity_on_hand"],
    errors="coerce"
)

inventory_df.head()

,item_number,description,quantity_on_hand
0,33412,10 BARRELL PUB FRUIT PUNCH,12.0
1,33413,10 BARRELL PUB ORIGINAL,12.0
2,12476,14TH STAR CITRANADE,16.0
3,125101,1809 BERLINER WEISSE,16.9
4,10602,1911 BLACK CHERRY,16.0


In [9]:
print("Inventory rows:", len(inventory_df))
print("Unique items:", inventory_df["item_number"].nunique())
print("Total quantity on hand:", round(inventory_df["quantity_on_hand"].sum(), 2))

inventory_df.describe()

Inventory rows: 2073
Unique items: 2073
Total quantity on hand: 179002.46


,quantity_on_hand
count,2073.000000
mean,86.349474
std,1792.969401
min,0.000000
25%,6.000000
50%,12.000000
75%,16.000000
max,59884.000000


In [10]:
inventory_df.to_csv("../data/cleaned/clean_inventory.csv", index=False)

print("Clean inventory data saved.")

Clean inventory data saved.


In [11]:
product_summary = pd.read_csv("../data/cleaned/product_summary.csv")

product_summary.head()

,inventory_number,description,total_quantity,total_revenue,total_cost,total_profit,transactions,profit_margin,revenue_rank,profit_rank,volume_rank
0,00102,HIGH LIFE 1/2 KG,5,464.95,365.0,99.95,3,0.2150,90.0,113.0,526.5
1,00107,HIGH LIFE 7OZ NR CASE,4,69.96,54.6,15.36,4,0.2196,617.0,735.0,639.0
2,00124*12,HIGH LIFE 12CAN 12PK,18,233.82,162.0,71.82,17,0.3072,211.0,173.0,132.0
3,00130,HIGH LIFE 30PK,70,1636.94,1288.0,348.94,29,0.2132,22.0,25.0,28.5
4,00144,HIGH LIFE 12NR 2/12PK CASE,17,400.83,317.9,82.93,13,0.2069,114.0,146.0,140.5


In [13]:
product_summary["inventory_number"] = product_summary["inventory_number"].astype(str)
inventory_df["item_number"] = inventory_df["item_number"].astype(str)

inventory_sales = product_summary.merge(
    inventory_df,
    left_on="inventory_number",
    right_on="item_number",
    how="left"
)

inventory_sales.head()

,inventory_number,description_x,total_quantity,total_revenue,total_cost,total_profit,transactions,profit_margin,revenue_rank,profit_rank,volume_rank,item_number,description_y,quantity_on_hand
0,00102,HIGH LIFE 1/2 KG,5,464.95,365.0,99.95,3,0.2150,90.0,113.0,526.5,NaN,NaN,NaN
1,00107,HIGH LIFE 7OZ NR CASE,4,69.96,54.6,15.36,4,0.2196,617.0,735.0,639.0,NaN,NaN,NaN
2,00124*12,HIGH LIFE 12CAN 12PK,18,233.82,162.0,71.82,17,0.3072,211.0,173.0,132.0,NaN,NaN,NaN
3,00130,HIGH LIFE 30PK,70,1636.94,1288.0,348.94,29,0.2132,22.0,25.0,28.5,NaN,NaN,NaN
4,00144,HIGH LIFE 12NR 2/12PK CASE,17,400.83,317.9,82.93,13,0.2069,114.0,146.0,140.5,NaN,NaN,NaN


In [14]:
print("Products in sales summary:", len(product_summary))
print("Products matched with inventory:", inventory_sales["quantity_on_hand"].notna().sum())
print("Products missing inventory:", inventory_sales["quantity_on_hand"].isna().sum())

Products in sales summary: 1629
Products matched with inventory: 210
Products missing inventory: 1419


In [27]:
import re

def clean_description(text):
    text = str(text).upper()

    # Remove punctuation
    text = re.sub(r"[^A-Z0-9 ]", "", text)

    # Remove extra spaces
    text = " ".join(text.split())

    return text

product_summary["merge_description"] = (
    product_summary["description"]
    .apply(clean_description)
)
inventory_df["merge_description"] = (
    inventory_df["description"]
    .apply(clean_description)
)

In [28]:
inventory_sales = product_summary.merge(
    inventory_df,
    on="merge_description",
    how="left",
    suffixes=("_sales", "_inventory")
)

inventory_sales.head()

,inventory_number,description_sales,total_quantity,total_revenue,total_cost,total_profit,transactions,profit_margin,revenue_rank,profit_rank,volume_rank,match_key_sales,merge_description,item_number,description_inventory,quantity_on_hand,match_key_inventory
0,00102,HIGH LIFE 1/2 KG,5,464.95,365.0,99.95,3,0.2150,90.0,113.0,526.5,102,HIGH LIFE 12 KG,NaN,NaN,NaN,NaN
1,00107,HIGH LIFE 7OZ NR CASE,4,69.96,54.6,15.36,4,0.2196,617.0,735.0,639.0,107,HIGH LIFE 7OZ NR CASE,NaN,NaN,NaN,NaN
2,00124*12,HIGH LIFE 12CAN 12PK,18,233.82,162.0,71.82,17,0.3072,211.0,173.0,132.0,124*12,HIGH LIFE 12CAN 12PK,NaN,NaN,NaN,NaN
3,00130,HIGH LIFE 30PK,70,1636.94,1288.0,348.94,29,0.2132,22.0,25.0,28.5,130,HIGH LIFE 30PK,NaN,NaN,NaN,NaN
4,00144,HIGH LIFE 12NR 2/12PK CASE,17,400.83,317.9,82.93,13,0.2069,114.0,146.0,140.5,144,HIGH LIFE 12NR 212PK CASE,NaN,NaN,NaN,NaN


In [30]:
print("Sales products:", len(product_summary))
print("Matched products:", inventory_sales["quantity_on_hand"].notna().sum())
print("Missing products:", inventory_sales["quantity_on_hand"].isna().sum())

match_rate = inventory_sales["quantity_on_hand"].notna().mean()

print(f"Match rate: {match_rate:.2%}")

Sales products: 1629
Matched products: 89
Missing products: 1571
Match rate: 5.36%


In [31]:
missing_inventory = inventory_sales[
    inventory_sales["quantity_on_hand"].isna()
][
    [
        "inventory_number",
        "description_sales",
        "total_quantity",
        "total_revenue"
    ]
].sort_values("total_revenue", ascending=False)

missing_inventory.head(25)

,inventory_number,description_sales,total_quantity,total_revenue
20,00236,LITE 36PK,281,8822.02
85,10230U,MICHELOB ULTRA 30PK,176,5698.37
1139,650941,SUPER LYTE VARIETY 3/8 CAS,85,4899.15
737,35930,COORS LT 30PK,132,3918.68
1140,650941*8,SUPER LYTE VARIETY 8PK,185,3870.55
8,00202,LITE 1/2 KG,23,3052.77
472,21024,YUENGLING LAGER 12OZ CAN 2,131,3019.49
19,00230,LITE 30PK,92,2734.08
717,35742,CORONA 12NR LOOSE CASE,72,2656.28
740,35944,COORS LIGHT 12NR LOOSE CAS,103,2643.23


In [33]:
inventory_sales["quantity_on_hand"] = inventory_sales["quantity_on_hand"].fillna(0)
inventory_sales["inventory_coverage_ratio"] = (
    inventory_sales["quantity_on_hand"] / inventory_sales["total_quantity"]
).round(2)

inventory_sales.head()

,inventory_number,description_sales,total_quantity,total_revenue,total_cost,total_profit,transactions,profit_margin,revenue_rank,profit_rank,volume_rank,match_key_sales,merge_description,item_number,description_inventory,quantity_on_hand,match_key_inventory,inventory_coverage_ratio
0,00102,HIGH LIFE 1/2 KG,5,464.95,365.0,99.95,3,0.2150,90.0,113.0,526.5,102,HIGH LIFE 12 KG,NaN,NaN,0.0,NaN,0.0
1,00107,HIGH LIFE 7OZ NR CASE,4,69.96,54.6,15.36,4,0.2196,617.0,735.0,639.0,107,HIGH LIFE 7OZ NR CASE,NaN,NaN,0.0,NaN,0.0
2,00124*12,HIGH LIFE 12CAN 12PK,18,233.82,162.0,71.82,17,0.3072,211.0,173.0,132.0,124*12,HIGH LIFE 12CAN 12PK,NaN,NaN,0.0,NaN,0.0
3,00130,HIGH LIFE 30PK,70,1636.94,1288.0,348.94,29,0.2132,22.0,25.0,28.5,130,HIGH LIFE 30PK,NaN,NaN,0.0,NaN,0.0
4,00144,HIGH LIFE 12NR 2/12PK CASE,17,400.83,317.9,82.93,13,0.2069,114.0,146.0,140.5,144,HIGH LIFE 12NR 212PK CASE,NaN,NaN,0.0,NaN,0.0


In [34]:
def inventory_status(row):
    if row["total_quantity"] <= 0:
        return "Review"
    elif row["quantity_on_hand"] == 0:
        return "Out / Missing Inventory"
    elif row["inventory_coverage_ratio"] < 0.25:
        return "Critical Low Stock"
    elif row["inventory_coverage_ratio"] < 0.75:
        return "Low Stock"
    elif row["inventory_coverage_ratio"] <= 2.00:
        return "Healthy Stock"
    else:
        return "Potential Overstock"

inventory_sales["inventory_status"] = inventory_sales.apply(inventory_status, axis=1)

inventory_sales["inventory_status"].value_counts()

inventory_status
Out / Missing Inventory    1565
Potential Overstock          46
Healthy Stock                18
Low Stock                    13
Critical Low Stock           11
Review                        7
Name: count, dtype: int64

In [35]:
reorder_candidates = inventory_sales[
    inventory_sales["inventory_status"].isin(
        ["Out / Missing Inventory", "Critical Low Stock", "Low Stock"]
    )
].sort_values("total_revenue", ascending=False)

reorder_candidates[
    [
        "description_sales",
        "total_quantity",
        "quantity_on_hand",
        "inventory_coverage_ratio",
        "total_revenue",
        "total_profit",
        "inventory_status"
    ]
].head(25)

,description_sales,total_quantity,quantity_on_hand,inventory_coverage_ratio,total_revenue,total_profit,inventory_status
20,LITE 36PK,281,0.0,0.00,8822.02,2078.02,Out / Missing Inventory
85,MICHELOB ULTRA 30PK,176,0.0,0.00,5698.37,1515.97,Out / Missing Inventory
1139,SUPER LYTE VARIETY 3/8 CAS,85,0.0,0.00,4899.15,1074.15,Out / Missing Inventory
737,COORS LT 30PK,132,0.0,0.00,3918.68,860.88,Out / Missing Inventory
1140,SUPER LYTE VARIETY 8PK,185,0.0,0.00,3870.55,1097.40,Out / Missing Inventory
8,LITE 1/2 KG,23,0.0,0.00,3052.77,499.77,Out / Missing Inventory
472,YUENGLING LAGER 12OZ CAN 2,131,0.0,0.00,3019.49,609.09,Out / Missing Inventory
19,LITE 30PK,92,0.0,0.00,2734.08,581.28,Out / Missing Inventory
717,CORONA 12NR LOOSE CASE,72,0.0,0.00,2656.28,526.13,Out / Missing Inventory
740,COORS LIGHT 12NR LOOSE CAS,103,0.0,0.00,2643.23,521.43,Out / Missing Inventory
